In [ ]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Direktori proyek
PROJECT_DIR = Path.cwd().parent

# Dataset Malaysia v7 yang sudah difilter
MALAYSIA_DIR = (
    PROJECT_DIR
    / "datasets"
    / "processed"
    / "dataset-malaysia-v7-filtered"
)

# Folder gambar dan label
TRAIN_IMAGE_DIR = MALAYSIA_DIR / "train" / "images"
TRAIN_LABEL_DIR = MALAYSIA_DIR / "train" / "labels"

TEST_IMAGE_DIR = MALAYSIA_DIR / "test" / "images"
TEST_LABEL_DIR = MALAYSIA_DIR / "test" / "labels"

# Class mapping
CLASS_NAMES = {
    0: "Unripe",
    1: "Ripe"
}

# Pemeriksaan direktori
for folder in [
    TRAIN_IMAGE_DIR,
    TRAIN_LABEL_DIR,
    TEST_IMAGE_DIR,
    TEST_LABEL_DIR
]:
    assert folder.exists(), f"Folder tidak ditemukan: {folder}"

print("Konfigurasi visual audit berhasil.")
print("Train images:", len(list(TRAIN_IMAGE_DIR.iterdir())))
print("Test images:", len(list(TEST_IMAGE_DIR.iterdir())))
print("Class mapping:", CLASS_NAMES)

In [ ]:
def count_boxes_per_image(image_dir, label_dir):
    results = []

    image_paths = [
        path for path in image_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]

    for image_path in image_paths:
        label_path = label_dir / f"{image_path.stem}.txt"

        with open(label_path, "r", encoding="utf-8") as file:
            box_count = sum(1 for line in file if line.strip())

        results.append({
            "image": image_path.name,
            "box_count": box_count
        })

    return pd.DataFrame(results)


train_box_df = count_boxes_per_image(
    TRAIN_IMAGE_DIR,
    TRAIN_LABEL_DIR
)

test_box_df = count_boxes_per_image(
    TEST_IMAGE_DIR,
    TEST_LABEL_DIR
)

print("TRAIN - Distribusi jumlah bounding box:")
display(
    train_box_df["box_count"]
    .value_counts()
    .sort_index()
    .rename_axis("jumlah_box")
    .to_frame("jumlah_gambar")
)

print("TEST - Distribusi jumlah bounding box:")
display(
    test_box_df["box_count"]
    .value_counts()
    .sort_index()
    .rename_axis("jumlah_box")
    .to_frame("jumlah_gambar")
)

In [ ]:
def visualize_image(image_path, label_path, ax):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    ax.imshow(image)
    ax.set_title(image_path.name, fontsize=9)
    ax.axis("off")

    if not label_path.exists():
        ax.set_title(f"{image_path.name} (LABEL MISSING)", color="red")
        return

    with open(label_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    for line in lines:
        parts = line.strip().split()

        if len(parts) != 5:
            continue

        class_id = int(parts[0])
        x_center, y_center, box_width, box_height = map(float, parts[1:])

        # Konversi koordinat YOLO ke pixel
        x_center *= width
        y_center *= height
        box_width *= width
        box_height *= height

        x_min = x_center - box_width / 2
        y_min = y_center - box_height / 2

        rect = patches.Rectangle(
            (x_min, y_min),
            box_width,
            box_height,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(rect)

        class_name = CLASS_NAMES.get(class_id, f"Unknown {class_id}")

        ax.text(
            x_min,
            max(0, y_min - 5),
            class_name,
            color="black",
            fontsize=9,
            bbox=dict(facecolor="lime", alpha=0.8)
        )


print("Fungsi visualisasi berhasil dibuat.")

In [ ]:
def show_random_samples(image_dir, label_dir, title, n=6, seed=42):
    image_paths = sorted([
        path for path in image_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ])

    if not image_paths:
        print(f"Tidak ada gambar di {image_dir}")
        return

    random.seed(seed)
    selected = random.sample(image_paths, min(n, len(image_paths)))

    cols = 3
    rows = (len(selected) + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = list(axes.flat) if hasattr(axes, "flat") else [axes]

    for ax, image_path in zip(axes, selected):
        label_path = label_dir / f"{image_path.stem}.txt"
        visualize_image(image_path, label_path, ax)

    for ax in axes[len(selected):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


show_random_samples(
    TRAIN_IMAGE_DIR,
    TRAIN_LABEL_DIR,
    "Malaysia v7 - Random Training Samples",
    n=6
)

show_random_samples(
    TEST_IMAGE_DIR,
    TEST_LABEL_DIR,
    "Malaysia v7 - Random Testing Samples",
    n=6,
    seed=100
)

In [ ]:
from pathlib import Path
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

PROJECT_DIR = Path.cwd().parent

MALAYSIA_DIR = (
    PROJECT_DIR
    / "datasets"
    / "processed"
    / "dataset-malaysia-v7-filtered"
)

TRAIN_IMAGE_DIR = MALAYSIA_DIR / "train" / "images"
TRAIN_LABEL_DIR = MALAYSIA_DIR / "train" / "labels"

TEST_IMAGE_DIR = MALAYSIA_DIR / "test" / "images"
TEST_LABEL_DIR = MALAYSIA_DIR / "test" / "labels"

CLASS_NAMES = {
    0: "Unripe",
    1: "Ripe"
}

for folder in [
    TRAIN_IMAGE_DIR,
    TRAIN_LABEL_DIR,
    TEST_IMAGE_DIR,
    TEST_LABEL_DIR
]:
    assert folder.exists(), f"Folder tidak ditemukan: {folder}"

print("Konfigurasi berhasil.")

In [ ]:
multi_box_images = train_box_df[
    train_box_df["box_count"] > 1
].sort_values("box_count", ascending=False)

print("Jumlah gambar dengan lebih dari satu box:", len(multi_box_images))
display(multi_box_images)

In [ ]:
# Ambil 12 gambar pertama dari daftar multi-box
batch = multi_box_images.head(12)

fig, axes = plt.subplots(4, 3, figsize=(18, 22))
axes = axes.flatten()

for ax, (_, row) in zip(axes, batch.iterrows()):
    image_path = TRAIN_IMAGE_DIR / row["image"]
    label_path = TRAIN_LABEL_DIR / f"{image_path.stem}.txt"

    visualize_image(image_path, label_path, ax)

    ax.set_title(
        f"{image_path.name[:35]}...\n"
        f"Jumlah box: {row['box_count']}",
        fontsize=9
    )

plt.suptitle("Malaysia v7 - Multi-Bounding Box Audit (Batch 1)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
batch = multi_box_images.iloc[12:24]

fig, axes = plt.subplots(4, 3, figsize=(18, 22))
axes = axes.flatten()

for ax, (_, row) in zip(axes, batch.iterrows()):
    image_path = TRAIN_IMAGE_DIR / row["image"]
    label_path = TRAIN_LABEL_DIR / f"{image_path.stem}.txt"

    visualize_image(image_path, label_path, ax)

    ax.set_title(
        f"{image_path.name[:35]}...\n"
        f"Jumlah box: {row['box_count']}",
        fontsize=9
    )

plt.suptitle("Malaysia v7 - Multi-Bounding Box Audit (Batch 2)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
batch = multi_box_images.iloc[24:36]

fig, axes = plt.subplots(4, 3, figsize=(18, 22))
axes = axes.flatten()

for ax, (_, row) in zip(axes, batch.iterrows()):
    image_path = TRAIN_IMAGE_DIR / row["image"]
    label_path = TRAIN_LABEL_DIR / f"{image_path.stem}.txt"

    visualize_image(image_path, label_path, ax)

    ax.set_title(
        f"{image_path.name[:35]}...\n"
        f"Jumlah box: {row['box_count']}",
        fontsize=9
    )

plt.suptitle("Malaysia v7 - Multi-Bounding Box Audit (Batch 3)", fontsize=16)
plt.tight_layout()
plt.show()